# Lección 6.4 — XGBoost, LightGBM y CatBoost: qué cambia entre ellos

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 6.3 (Gradient Boosting), Lección 2.3 (codificación de categorías), Lección 2.2 (datos faltantes)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué existen XGBoost, LightGBM y CatBoost si scikit-learn ya tenía Gradient Boosting.
- Describir los tres trucos que más cambian: **histogramas**, **forma de crecer el árbol** y **manejo nativo de faltantes y categorías**.
- Traducir los hiperparámetros principales entre las cuatro librerías (`learning_rate`, `n_estimators`, `max_depth`/`num_leaves`, `subsample`, `colsample`, regularización).
- Ajustar un boosting moderno con **parada temprana** usando `HistGradientBoostingClassifier` de scikit-learn.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 **Sobre las librerías.** XGBoost, LightGBM y CatBoost se instalan aparte (`pip install xgboost lightgbm catboost`). Para que este notebook funcione en cualquier lado, el código ejecutable usa **`HistGradientBoostingClassifier`** de scikit-learn, que está inspirado directamente en LightGBM y comparte sus ideas clave. Verás el código equivalente de las tres librerías en celdas marcadas **"📦 Para Colab"**: cópialas en Google Colab (donde ya vienen instaladas) si quieres probarlas.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (GradientBoostingClassifier, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn.datasets import make_classification, fetch_openml

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el mismo motor, tres escuderías

En la Lección 6.3 viste que Gradient Boosting es una idea sencilla: **árboles pequeños que persiguen residuos**. Es como un motor de combustión: la idea es la misma en todos los autos.

Pero en una carrera de Fórmula 1, cada escudería mejora el **mismo motor** a su manera: una lo hace más potente, otra más ligero, otra más fácil de manejar. Con el boosting pasó igual:

| Librería | Año / creador | Su "obsesión" | En una frase |
|---|---|---|---|
| **XGBoost** | 2014 · Tianqi Chen (Univ. de Washington) | **Regularización y robustez** | El que popularizó el boosting ganando decenas de competencias de Kaggle |
| **LightGBM** | 2017 · Microsoft | **Velocidad** con datos grandes | Histogramas + árboles que crecen "por la hoja más prometedora" |
| **CatBoost** | 2017 · Yandex | **Variables categóricas** y buenos valores por defecto | Árboles simétricos y un truco anti-fuga para codificar categorías |
| **HistGradientBoosting** | 2019 · scikit-learn | Traer las ideas de LightGBM a scikit-learn | Lo que usaremos en este notebook |

Las cuatro hacen **lo mismo** que viste en la Lección 6.3. Las diferencias están en **cómo** buscan las preguntas, **cómo** crecen los árboles y **qué tipos de datos** aceptan sin preparación.

### ✋ Ejercicio 1 — ¿Dónde está el cuello de botella?
Recuerda cómo un árbol elige una pregunta (Lección 5.4): prueba **todas las características** y **todos los puntos de corte posibles**.

Tienes 1 millón de clientes y 50 columnas numéricas, casi todas con valores distintos para cada cliente (ingreso, saldo…). Para la **primera** pregunta del **primer** árbol, ¿cuántos puntos de corte habría que probar, aproximadamente? ¿Y para 500 árboles con 30 preguntas cada uno?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Unos **50 millones** de cortes para una sola pregunta (≈1 millón de valores distintos × 50 columnas). Con 500 árboles × 30 preguntas, del orden de **750 mil millones**. Es muchísimo trabajo, y por eso el Gradient Boosting clásico se vuelve lentísimo con datos grandes. El primer gran truco de las librerías modernas ataca justo esto.
</details>

---
## 2. 📖 Truco 1: histogramas (agrupar antes de preguntar)

En lugar de probar cada valor exacto, se agrupan los valores de cada columna en **cubetas** (*bins*), por ejemplo **255**, como las barras de un histograma. Luego solo se prueban cortes **entre cubetas**.

```
Valores exactos (1 millón):  12,031 · 12,032 · 12,047 · 12,050 · ... · 98,410   → 1,000,000 cortes posibles
Con histograma (255 cubetas): [12k–13k] [13k–14k] [14k–15k] ... [97k–99k]        →       254 cortes posibles
```

Se pierde un poquito de precisión en el punto de corte (¿cortar en 12,047 o en 12,050?), pero casi nunca importa, y la velocidad se multiplica. **LightGBM**, **XGBoost** (con `tree_method="hist"`, su valor por defecto desde la versión 2.0) y **HistGradientBoosting** usan este truco.

### 🔮 Ejercicio 2 — Carrera de velocidad
Vamos a entrenar 100 árboles con 20,000 ejemplos y 10 columnas usando:
- `GradientBoostingClassifier` (clásico, valores exactos).
- `HistGradientBoostingClassifier` (con histogramas).

¿Cuántas veces más rápido crees que será el de histogramas? ¿Perderá exactitud?

✍️ *Mi apuesta:*

In [ ]:
X_rap, y_rap = make_classification(n_samples=25_000, n_features=10, n_informative=6, random_state=0)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_rap, y_rap, test_size=5_000, random_state=0)

for nombre, modelo in [
    ("Clásico (valores exactos)", GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=0)),
    ("Con histogramas          ", HistGradientBoostingClassifier(max_iter=100, max_depth=5, early_stopping=False, random_state=0)),
]:
    inicio = time.perf_counter()
    modelo.fit(Xr_tr, yr_tr)
    segundos = time.perf_counter() - inicio
    print(f"{nombre}: {segundos:5.1f} s · exactitud en prueba = {modelo.score(Xr_te, yr_te):.3f}")

valores_distintos = len(np.unique(Xr_tr[:, 0]))
print(f"\nLa columna 0 tiene {valores_distintos:,} valores distintos; el histograma la resume en 255 cubetas.")

<details><summary><b>👀 Ver respuesta</b></summary>

En nuestras ejecuciones, el clásico tardó entre **≈8 y ≈43 s** y el de histogramas entre **≈0.5 y ≈13 s** (según qué tan ocupada estaba la computadora): de **3 a más de 20 veces** menos, con la **misma** exactitud (≈0.906 contra ≈0.907). En tu computadora los segundos serán otros, pero el de histogramas debe ganar con claridad, y la ventaja **crece** con más filas: con millones, es la diferencia entre minutos y horas.

¿Por qué tanta diferencia? La columna 0 tiene **20,000** valores distintos (uno por ejemplo): el histograma la reduce a 255 cubetas, unas **80 veces** menos cortes que probar en cada pregunta.
</details>

---
## 3. 📖 Truco 2: cómo crece cada árbol

Las librerías no solo buscan las preguntas más rápido: también **eligen qué hoja dividir** de forma distinta.

```
POR NIVELES (XGBoost por defecto)   POR HOJAS (LightGBM, HistGB)        SIMÉTRICO (CatBoost)
divide TODAS las hojas de un nivel   divide la hoja que más mejora,      la MISMA pregunta en todo
antes de bajar al siguiente          esté donde esté                     el nivel
          ●                                  ●                                  ●  ¿x1>5?
       ┌──┴──┐                            ┌──┴──┐                            ┌──┴──┐
       ●     ●                            ●     ○                         ¿x2>3? ¿x2>3?
     ┌─┴─┐ ┌─┴─┐                        ┌─┴─┐                             ┌─┴─┐ ┌─┴─┐
     ○   ○ ○   ○                        ●   ○                             ○   ○ ○   ○
                                      ┌─┴─┐
                                      ○   ○
 Árbol "ancho" y equilibrado.         Árbol "profundo" y desigual:        Árbol muy regular: rapidísimo
 Controlas con max_depth.             más preciso con las mismas hojas,   para predecir y difícil de
                                      pero sobreajusta si te descuidas.   sobreajustar.
                                      Controlas con num_leaves.
```

`DecisionTreeClassifier` de scikit-learn puede crecer de las dos primeras formas: con `max_depth` crece por niveles, y con `max_leaf_nodes` (sin `max_depth`) crece **por hojas**, eligiendo siempre la división que más mejora.

### 🔮 Ejercicio 3 — Mismas hojas, distinta forma
Entrenamos dos árboles con unas **32 hojas** cada uno: uno por niveles (`max_depth=5` → como máximo 2⁵ = 32 hojas) y otro por hojas (`max_leaf_nodes=32`).
1. ¿Cuál será más **profundo**?
2. ¿Cuál será más **exacto** en prueba?

✍️ *Mis predicciones:*

In [ ]:
por_niveles = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xr_tr, yr_tr)
por_hojas = DecisionTreeClassifier(max_leaf_nodes=32, random_state=0).fit(Xr_tr, yr_tr)

for nombre, arbol in [("Por niveles (max_depth=5)    ", por_niveles), ("Por hojas (max_leaf_nodes=32)", por_hojas)]:
    print(f"{nombre}: hojas={arbol.get_n_leaves()}  profundidad={arbol.get_depth()}  prueba={arbol.score(Xr_te, yr_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El árbol **por hojas** llega a profundidad **9**, contra 5 del árbol por niveles, con prácticamente las mismas hojas (32 contra 31).
2. También es más **exacto**: ≈0.870 contra ≈0.836. En lugar de "gastar" hojas dividiendo zonas que ya estaban bien, las concentró donde más mejoraba el resultado.

Por eso en **LightGBM** la perilla principal es `num_leaves` (y `max_depth` queda como freno de seguridad), y en **HistGradientBoosting** es `max_leaf_nodes` (por defecto 31). ⚠️ Con pocos datos, crecer por hojas puede generar ramas muy profundas que memorizan: por eso LightGBM recomienda limitar `num_leaves` y subir `min_child_samples` en datos pequeños.
</details>

---
## 4. 📖 Truco 3: faltantes sin rellenar

En la Lección 2.2 aprendiste a **rellenar** (imputar) los datos faltantes antes de entrenar. Las librerías modernas permiten saltarte ese paso: en cada división, el árbol **aprende hacia qué lado mandar los faltantes** (prueba los dos lados y se queda con el que más mejora).

¿Y eso qué ventaja tiene? Que a veces **que falte un dato es información**. Imagina una financiera donde algunos solicitantes **se niegan a declarar su ingreso**… y justo esos pagan peor.

In [ ]:
rng = np.random.default_rng(0)
n = 5_000
ingreso = rng.lognormal(10, 0.5, n)
deuda_pct = rng.uniform(0, 1, n)
edad = rng.integers(18, 70, n)
no_declara = rng.random(n) < 0.15                      # 15 % no declara su ingreso
riesgo = -1 - 2.4 * (np.log(ingreso) - 10) + 2 * deuda_pct - 0.02 * (edad - 40) + 1.8 * no_declara
impago = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

solicitudes = pd.DataFrame({"ingreso": np.where(no_declara, np.nan, ingreso), "deuda_pct": deuda_pct, "edad": edad})
print(f"Faltantes en 'ingreso': {solicitudes.ingreso.isna().mean():.0%}")
print(f"Impago entre quienes SÍ declaran: {impago[~no_declara].mean():.0%} · entre quienes NO declaran: {impago[no_declara].mean():.0%}")
solicitudes.head()

### 🔮 Ejercicio 4 — ¿Qué hacemos con los faltantes?
Primero comprobamos que el Gradient Boosting clásico **ni siquiera acepta** faltantes. Luego comparamos tres estrategias, todas con el mismo modelo (HistGradientBoosting) y validación cruzada:
- **A:** quitar la columna `ingreso` (la salida "fácil").
- **B:** rellenar el ingreso faltante con el **promedio** (Lección 2.2).
- **C:** pasarle los datos **tal cual**, con faltantes.

¿Cuál crees que gana? ¿Habrá mucha diferencia entre B y C?

✍️ *Mi apuesta:*

In [ ]:
try:
    GradientBoostingClassifier().fit(solicitudes, impago)
except ValueError as e:
    print("❌ Gradient Boosting clásico con faltantes:", str(e).splitlines()[0])

modelo = HistGradientBoostingClassifier(random_state=0)
estrategias = {
    "A: quitar la columna        ": (modelo, solicitudes.drop(columns="ingreso")),
    "B: rellenar con el promedio ": (make_pipeline(SimpleImputer(strategy="mean"), modelo), solicitudes),
    "C: faltantes nativos        ": (modelo, solicitudes),
}
for nombre, (m, datos) in estrategias.items():
    auc = cross_val_score(m, datos, impago, cv=5, scoring="roc_auc").mean()
    print(f"{nombre} → AUC en validación cruzada: {auc:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **A (quitar la columna)** es un desastre: AUC ≈**0.58**, casi como adivinar. El ingreso era la variable más útil.
- **B (rellenar con el promedio)** y **C (nativo)** empatan: AUC ≈**0.780** las dos.

¿Por qué empatan? Porque a todos los que no declararon se les puso **el mismo número exacto** (el promedio), y un árbol puede aislar ese valor con dos cortes: en la práctica, "descubre" quiénes no declararon. La ventaja de C no es la exactitud sino que **no tuviste que hacer nada** y no hay riesgo de equivocarte al rellenar (p. ej. calcular el promedio con todos los datos y provocar una fuga, Lección 2.6). Ojo: un modelo **lineal** sí perdería la señal al rellenar con el promedio.

(Usamos AUC, Lección 4.3, porque mide qué tan bien ordena el modelo a los clientes por riesgo.)

💡 Si quieres usar un modelo que no acepta faltantes, puedes conservar esa información con `SimpleImputer(add_indicator=True)`, que agrega una columna "faltaba / no faltaba".
</details>

---
## 5. 📖 Truco 4: categorías sin one-hot

Con una variable categórica de **muchos** valores (ciudad, producto, código postal), el **one-hot** (Lección 2.3) crea una columna por valor: con 60 ciudades, 60 columnas casi vacías. Un árbol necesitaría muchas preguntas "¿es Monterrey? ¿es Lima?…" para agruparlas.

Las librerías modernas tratan las categorías de forma **nativa**:
- **LightGBM** y **HistGradientBoosting**: en cada división, ordenan las categorías según su efecto y buscan el mejor **grupo** de categorías ("¿ciudad ∈ {Lima, Quito, Cali}?") en una sola pregunta.
- **CatBoost**: convierte cada categoría en un número usando el **promedio del objetivo** (*target encoding*), pero calculado de forma "ordenada": para cada fila usa solo las filas **anteriores**, así la fila nunca ve su propia respuesta. Es un truco **anti-fuga** (Lección 2.6).
- **XGBoost**: soporte nativo más reciente (`enable_categorical=True`).

Probémoslo con ventas de 20,000 tiendas en **60 ciudades**, donde cada ciudad tiene su propio efecto.

In [ ]:
rng_v = np.random.default_rng(1)
n_v, n_ciudades = 20_000, 60
efecto_ciudad = rng_v.normal(0, 1, n_ciudades)
id_ciudad = rng_v.integers(0, n_ciudades, n_v)
tiendas = pd.DataFrame({
    "ciudad": pd.Categorical([f"Ciudad_{c:02d}" for c in id_ciudad]),   # tipo 'category' de pandas
    "m2": rng_v.uniform(20, 300, n_v),
    "promocion": rng_v.integers(0, 2, n_v),
})
ventas = 2 * efecto_ciudad[id_ciudad] + 0.01 * tiendas.m2 + tiendas.promocion + rng_v.normal(0, 1, n_v)
tiendas.head()

### 🔮 Ejercicio 5 — Tres formas de darle la ciudad al modelo
1. **One-hot:** 60 columnas de 0/1.
2. **Ordinal:** cada ciudad es un número (Ciudad_00 = 0, Ciudad_01 = 1…), un orden **arbitrario**.
3. **Nativa:** le decimos al modelo que `ciudad` es categórica (`categorical_features="from_dtype"`).

¿Cuál esperas que tenga mejor R² (Lección 4.1)? ¿Cuál será la más lenta?

✍️ *Mi apuesta:*

In [ ]:
formas = {
    "1. One-hot": make_pipeline(make_column_transformer((OneHotEncoder(sparse_output=False), ["ciudad"]), remainder="passthrough"),
                                HistGradientBoostingRegressor(random_state=0)),
    "2. Ordinal": make_pipeline(make_column_transformer((OrdinalEncoder(), ["ciudad"]), remainder="passthrough"),
                                HistGradientBoostingRegressor(random_state=0)),
    "3. Nativa ": HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0),
}
for nombre, m in formas.items():
    inicio = time.perf_counter()
    r2 = cross_val_score(m, tiendas, ventas, cv=5, scoring="r2").mean()
    print(f"{nombre}: R² = {r2:.3f} · {time.perf_counter() - inicio:5.1f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

Las tres formas quedan **muy parecidas** en R² (≈0.785–0.790): un boosting es lo bastante potente para arreglárselas incluso con un orden arbitrario, haciendo varios cortes. La **nativa** queda un pelín por delante, pero la gran diferencia está en el **tiempo**: en nuestras ejecuciones, la nativa fue siempre la más rápida (≈0.4–1.4 s) y el **one-hot** el más lento (≈4–7 s), porque tiene que trabajar con 62 columnas en lugar de 3. Con miles de categorías (productos, códigos postales), el one-hot se vuelve inmanejable y la forma nativa es la única razonable.
</details>

---
## 6. 📖 Hiperparámetros: el diccionario de traducción

Las cuatro librerías tienen **las mismas perillas** con **nombres distintos**. Esta tabla es tu diccionario (en XGBoost y LightGBM, la "API de scikit-learn", que es la más usada):

| Idea | XGBoost (`XGBClassifier`) | LightGBM (`LGBMClassifier`) | CatBoost (`CatBoostClassifier`) | scikit-learn (`HistGradientBoosting…`) |
|---|---|---|---|---|
| Tasa de aprendizaje | `learning_rate` (= `eta`), def. 0.3 | `learning_rate`, def. 0.1 | `learning_rate`, automático | `learning_rate`, def. 0.1 |
| Número de árboles | `n_estimators`, def. 100 | `n_estimators`, def. 100 | `iterations`, def. 1000 | `max_iter`, def. 100 |
| Tamaño del árbol | `max_depth`, def. 6 | `num_leaves`, def. 31 (+ `max_depth`) | `depth`, def. 6 | `max_leaf_nodes`, def. 31 (+ `max_depth`) |
| Mínimo por hoja | `min_child_weight`, def. 1 | `min_child_samples`, def. 20 | `min_data_in_leaf`, def. 1 | `min_samples_leaf`, def. 20 |
| Fracción de filas por árbol | `subsample`, def. 1 | `subsample` (= `bagging_fraction`), def. 1 | `subsample` | — (no disponible) |
| Fracción de columnas | `colsample_bytree`, def. 1 | `colsample_bytree` (= `feature_fraction`), def. 1 | `rsm` | `max_features` (por división), def. 1.0 |
| Regularización L2 | `reg_lambda`, def. 1 | `reg_lambda`, def. 0 | `l2_leaf_reg`, def. 3 | `l2_regularization`, def. 0 |
| Regularización L1 | `reg_alpha`, def. 0 | `reg_alpha`, def. 0 | — | — |
| Ganancia mínima para dividir | `gamma`, def. 0 | `min_split_gain`, def. 0 | — | — |
| Parada temprana | `early_stopping_rounds` | callback `early_stopping` | `early_stopping_rounds` | `early_stopping`, `n_iter_no_change` (def. 10) |

### Qué hace cada perilla

| Hiperparámetro | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|
| `learning_rate` | Aprende rápido, sobreajusta antes | Necesita más árboles, generaliza mejor | 0.05–0.1 |
| `n_estimators` / `max_iter` | Menos sesgo; demasiados → sobreajuste | Subajuste | Muchos (1,000+) **con parada temprana** |
| `max_depth` / `num_leaves` | Árboles más complejos, más interacciones | Más simples | depth 4–8 · leaves 15–63 |
| `min_child_samples` / `min_samples_leaf` | Hojas más "confiables", menos sobreajuste | Permite hojas diminutas | 20 (subir con datos ruidosos) |
| `subsample` | — | Más azar, regulariza y acelera | 0.7–0.9 |
| `colsample_bytree` / `max_features` | — | Árboles más distintos (como Random Forest) | 0.5–0.9 |
| `reg_lambda` / `l2_regularization` | Hojas con valores más moderados, menos sobreajuste | Menos freno | 0–10 |

> 💡 **Regularización** (castigar la complejidad) es la gran aportación de XGBoost: añade a la pérdida un "castigo" por hojas con valores extremos (L2, `reg_lambda`), por usar muchas hojas (`gamma`) o por valores distintos de cero (L1, `reg_alpha`). Verás la regularización a fondo en la Lección 7.1.

### 📖 Datos para practicar con las perillas
Usaremos datos reales: el **censo de Estados Unidos de 1994** (*Adult*), con 48,842 personas. El objetivo es predecir si una persona gana **más de 50,000 dólares al año**. Tiene columnas numéricas, **categóricas** (ocupación, estado civil…) y **faltantes**: el escenario ideal para un boosting moderno. (La primera vez se descarga, ≈2 MB.)

In [ ]:
censo = fetch_openml("adult", version=2, as_frame=True).frame
X_censo = censo.drop(columns=["class", "fnlwgt"])          # fnlwgt es un peso de muestreo, no una característica
y_censo = (censo["class"] == ">50K").astype(int)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_censo, y_censo, test_size=0.25, stratify=y_censo, random_state=0)

print(f"Filas: {len(censo):,} · % que gana >50K: {y_censo.mean():.1%}")
print("Columnas categóricas:", list(X_censo.select_dtypes("category").columns))
print("Faltantes por columna:", X_censo.isna().sum()[X_censo.isna().sum() > 0].to_dict())

### 🔮 Ejercicio 6 — Parada temprana: ¿cuántos árboles necesita?
Le damos permiso para usar hasta **2,000** árboles, pero con **parada temprana**: aparta un 10 % de los datos de entrenamiento como validación y se detiene si en 10 rondas seguidas no mejora. Antes de ejecutar: con `learning_rate=0.1`, ¿cuántos árboles crees que usará realmente? ¿Y con `learning_rate=0.03`?

✍️ *Mi apuesta:*

In [ ]:
for lr in [0.3, 0.1, 0.03]:        # ✏️
    m = HistGradientBoostingClassifier(learning_rate=lr, max_iter=2_000, early_stopping=True,
                                       n_iter_no_change=10, validation_fraction=0.1,
                                       categorical_features="from_dtype", random_state=0)
    inicio = time.perf_counter()
    m.fit(Xc_tr, yc_tr)
    print(f"learning_rate={lr:<5} → árboles usados: {m.n_iter_:>4} · puntuación de validación: {m.validation_score_[-11]:.4f}"
          f" · {time.perf_counter() - inicio:4.1f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

En nuestra ejecución: con `learning_rate=0.3` usó **34** árboles; con 0.1, **114**; con 0.03, **310**. Ninguno se acercó a los 2,000 permitidos: la parada temprana decidió sola cuándo detenerse. Y es la regla mnemotécnica de la Lección 6.3: golpes más cortos → más golpes. Las puntuaciones de validación quedaron muy cerca (≈−0.268, −0.266, −0.265), con una ligera ventaja para la tasa más baja, a cambio de más tiempo.

(`validation_score_` guarda la puntuación de validación en cada ronda; miramos la de 11 rondas antes del final, la mejor que encontró antes de rendirse. Por defecto esa puntuación es la **pérdida logarítmica cambiada de signo** (Lección 3.2): más cerca de 0 = mejor. En la Lección 7.2 verás la parada temprana en detalle.)
</details>

### ✏️ Ejercicio 7 — Tamaño del árbol y regularización
Ahora, con `learning_rate=0.1` y parada temprana, probamos combinaciones de `max_leaf_nodes` (el `num_leaves` de LightGBM) y `l2_regularization` (el `reg_lambda` de XGBoost), con validación cruzada de 3 pliegues. Cambia las listas y observa.

In [ ]:
HOJAS = [7, 31, 127]               # ✏️ max_leaf_nodes
L2 = [0.0, 10.0]                   # ✏️ l2_regularization

tabla = pd.DataFrame(index=HOJAS, columns=L2, dtype=float)
for hojas in HOJAS:
    for l2 in L2:
        m = HistGradientBoostingClassifier(max_leaf_nodes=hojas, l2_regularization=l2, learning_rate=0.1,
                                           max_iter=1_000, early_stopping=True, categorical_features="from_dtype", random_state=0)
        tabla.loc[hojas, l2] = cross_val_score(m, Xc_tr, yc_tr, cv=3, scoring="roc_auc").mean()
tabla.index.name, tabla.columns.name = "max_leaf_nodes", "l2_regularization"
tabla.round(4)

✋ ¿Qué tamaño de árbol funciona mejor? ¿Cuánto mueve el resultado la regularización?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Toda la tabla está entre **≈0.924 y ≈0.927** de AUC: diferencias de milésimas. Lo mejor es `max_leaf_nodes=31` (el valor por defecto); árboles muy grandes (127 hojas) quedan un poco por debajo. La regularización L2 casi no mueve nada aquí: con 36,000 filas y parada temprana, el modelo ya tenía poco espacio para sobreajustar.

Lección importante: con datos de buen tamaño, **los valores por defecto de un boosting moderno ya son muy buenos**. El ajuste fino da milésimas; las grandes ganancias suelen venir de mejores **datos** y **variables** (Lección 2.4). En datos pequeños o ruidosos, en cambio, estas perillas sí pesan más (Módulo 7).
</details>

### ✏️ Ejercicio 8 — Muestrear columnas (`max_features` ≈ `colsample`)
`max_features` hace que en cada división solo se consideren algunas columnas al azar, como `max_features` en Random Forest (Lección 6.2) o `colsample_bynode` en XGBoost. Prueba varios valores.

In [ ]:
for mf in [1.0, 0.5, 0.2]:          # ✏️
    m = HistGradientBoostingClassifier(max_features=mf, learning_rate=0.1, max_iter=1_000, early_stopping=True,
                                       categorical_features="from_dtype", random_state=0)
    print(f"max_features={mf}: AUC (validación cruzada) = {cross_val_score(m, Xc_tr, yc_tr, cv=3, scoring='roc_auc').mean():.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Los tres valores quedan prácticamente empatados (≈0.926–0.927 de AUC), con una mínima ventaja para usar todas las columnas. En este conjunto, con solo 12 columnas, sortear columnas no aporta: la diversidad que da es pequeña y a cambio cada árbol pierde información. En datos con **cientos** de columnas (o columnas muy redundantes), `colsample`/`max_features` suele ayudar más, y además **acelera** el entrenamiento.
</details>

---
## 7. 📦 Para Colab: el mismo modelo en las tres librerías

Copia esta celda en **Google Colab** (ahí las tres librerías ya vienen instaladas; en tu computadora necesitas `pip install xgboost lightgbm catboost`). Usa los datos del censo de la sección anterior. Fíjate en lo parecido que es el código: **la idea es la misma, cambian los nombres**.

```python
# 📦 PARA COLAB — no se ejecuta en este notebook
# !pip install xgboost lightgbm catboost      # solo si no están instaladas
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# Separamos una validación para la parada temprana
Xa, Xv, ya, yv = train_test_split(Xc_tr, yc_tr, test_size=0.1, stratify=yc_tr, random_state=0)
cats = list(Xc_tr.select_dtypes("category").columns)

# --- XGBoost: crece por niveles (max_depth), regularización L2 por defecto ---
xgb = XGBClassifier(n_estimators=2000, learning_rate=0.1, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                    tree_method="hist", enable_categorical=True,   # categorías nativas
                    early_stopping_rounds=50, eval_metric="auc")
xgb.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)

# --- LightGBM: crece por hojas (num_leaves) ---
lgbm = LGBMClassifier(n_estimators=2000, learning_rate=0.1, num_leaves=31,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                      min_child_samples=20, verbose=-1)
lgbm.fit(Xa, ya, eval_set=[(Xv, yv)], callbacks=[early_stopping(50, verbose=False)])

# --- CatBoost: árboles simétricos, categorías con codificación ordenada ---
# CatBoost quiere las categorías como texto y sin NaN en ellas
Xa_cb, Xv_cb, Xte_cb = (d.astype({c: str for c in cats}) for d in (Xa, Xv, Xc_te))
cb = CatBoostClassifier(iterations=2000, learning_rate=0.1, depth=6,
                        cat_features=cats, early_stopping_rounds=50, verbose=False)
cb.fit(Xa_cb, ya, eval_set=(Xv_cb, yv))

for nombre, modelo, Xprueba in [("XGBoost", xgb, Xc_te), ("LightGBM", lgbm, Xc_te), ("CatBoost", cb, Xte_cb)]:
    auc = roc_auc_score(yc_te, modelo.predict_proba(Xprueba)[:, 1])
    print(f"{nombre:<9} AUC en prueba = {auc:.4f}")
```

> 🔎 **Qué esperar:** con datos como estos, las tres librerías (y HistGradientBoosting) suelen quedar **a pocas milésimas** de AUC entre sí. Las diferencias reales aparecen en **velocidad**, en **datos con muchas categorías** (donde CatBoost brilla) y en **datos enormes** (donde LightGBM suele ser el más rápido).

### ✋ Ejercicio 9 — Traduce
Un compañero te pasa esta configuración de **XGBoost**:
`XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=4, colsample_bytree=0.7, reg_lambda=5)`

Escribe una configuración **aproximadamente equivalente** de `HistGradientBoostingClassifier`. ¿Qué perilla **no** tiene traducción directa?

✍️ *Mi traducción:*

<details><summary><b>👀 Ver respuesta</b></summary>

```python
HistGradientBoostingClassifier(max_iter=800, learning_rate=0.05, max_depth=4, max_leaf_nodes=None,
                               max_features=0.7, l2_regularization=5)
```
- `n_estimators` → `max_iter`; `reg_lambda` → `l2_regularization`.
- `max_depth=4` por niveles da hasta 16 hojas; en HistGB pones `max_depth=4` y quitas el límite de hojas (`max_leaf_nodes=None`) para imitarlo. Otra opción es `max_leaf_nodes=16` (mismo número de hojas, pero creciendo por hojas).
- `colsample_bytree` sortea columnas **una vez por árbol**; `max_features` lo hace **en cada división** (como `colsample_bynode`). Es parecido, no idéntico.
- Si hubiera `subsample`, **no** tendría traducción: HistGradientBoosting no muestrea filas.
- Los resultados no serán idénticos (cada librería calcula los histogramas y las hojas a su manera), pero deberían ser muy parecidos.
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| "XGBoost me dio peor que Random Forest" | Comparaste con valores por defecto y sin parada temprana | Parada temprana + ajustar `learning_rate`, profundidad y `min_child_*` (Módulo 7) |
| LightGBM sobreajusta con un dataset pequeño (pocos miles de filas) | Crecer por hojas genera ramas muy profundas | Bajar `num_leaves`, subir `min_child_samples`, poner `max_depth` |
| Puntuación de validación excelente, prueba decepcionante | Usaste el **conjunto de prueba** como `eval_set` de la parada temprana | La validación para parar debe salir del **entrenamiento** |
| Target encoding "a mano" da resultados irreales | Fuga: cada fila vio su propia respuesta al codificar | Codificación dentro de la validación cruzada, `TargetEncoder` de scikit-learn, o CatBoost |
| Una columna de categorías con códigos numéricos (código postal) se trata como número | El modelo cree que 06700 < 64000 significa algo | Declararla categórica (`category` en pandas) |
| Resultados distintos en cada ejecución | Azar en `subsample`/`colsample` sin semilla | Fijar `random_state` (o `seed`) |

### 🔮 Ejercicio 10 — El código postal
En el ejemplo de las tiendas, ¿qué crees que pasaría con la forma **ordinal** si los números de las ciudades **sí** tuvieran un orden con sentido (por ejemplo, ordenadas de menor a mayor efecto en ventas)?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Funcionaría casi tan bien como la nativa: con un orden con sentido, un solo corte ("¿ciudad ≤ 23?") separa ciudades de ventas bajas y altas. El problema del ordinal no es usar números, sino usar números en un **orden arbitrario**. Justamente, lo que hacen LightGBM y HistGradientBoosting con las categorías es **encontrar ese buen orden** automáticamente en cada división.
</details>

---
## 9. 📋 Ficha comparativa

| | XGBoost | LightGBM | CatBoost | HistGradientBoosting (sklearn) |
|---|---|---|---|---|
| **Crecimiento del árbol** | Por niveles (`max_depth`) | Por hojas (`num_leaves`) | Simétrico (`depth`) | Por hojas (`max_leaf_nodes`) |
| **Búsqueda de cortes** | Histogramas (`hist`) | Histogramas + muestreo inteligente (GOSS) | Histogramas | Histogramas (≤255 cubetas) |
| **Faltantes** | ✅ nativo | ✅ nativo | ✅ nativo (numéricos) | ✅ nativo |
| **Categorías** | ✅ (`enable_categorical`) | ✅ nativo | ✅✅ su especialidad (codificación ordenada) | ✅ (`categorical_features`) |
| **Regularización** | L1, L2, `gamma` | L1, L2, `min_split_gain` | L2 | L2 |
| **GPU** | ✅ | ✅ | ✅ | ❌ |
| **✅ Brilla en…** | Robustez, ecosistema enorme, documentación | Velocidad con datos grandes | Muchas categorías, buenos valores por defecto | Cero instalación, integra con pipelines de sklearn |
| **❌ Cuidado con…** | Muchas perillas | Sobreajuste en datos pequeños | Más lento de entrenar; predice muy rápido | Menos perillas (sin `subsample` de filas) |

## 10. 🧠 Resumen en 5 puntos
1. XGBoost, LightGBM, CatBoost y HistGradientBoosting son **el mismo Gradient Boosting** de la Lección 6.3, con mejoras de ingeniería.
2. **Histogramas:** agrupar valores en ≤255 cubetas acelera muchísimo la búsqueda de cortes sin perder casi nada.
3. **Forma del árbol:** por niveles (XGBoost), por hojas (LightGBM, HistGB) o simétrico (CatBoost); por eso la perilla de tamaño se llama `max_depth`, `num_leaves` o `depth`.
4. Aceptan **faltantes** y **categorías** de forma nativa; a veces eso es incluso **mejor** que imputar o hacer one-hot.
5. Receta: `learning_rate` 0.05–0.1, **muchos árboles con parada temprana** y ajustar tamaño del árbol, mínimo por hoja, muestreo y regularización.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál es la idea **principal** de los histogramas en el boosting moderno?
- a) Dibujar gráficas más bonitas · b) Agrupar los valores en cubetas para probar muchos menos cortes · c) Eliminar los faltantes

**2.** Pasas de `max_depth=6` en XGBoost a LightGBM. ¿Qué perilla controla principalmente el tamaño del árbol en LightGBM?
- a) `num_leaves` · b) `n_estimators` · c) `learning_rate`

**3.** ¿Qué hace CatBoost para codificar categorías **sin fuga**?
- a) One-hot siempre · b) Promedio del objetivo calculado solo con filas "anteriores" · c) Borra las categorías raras

**4.** Tu modelo usa parada temprana con el **conjunto de prueba** como validación. ¿Qué problema hay?
- a) Ninguno · b) La prueba deja de ser una evaluación honesta porque influyó en una decisión (cuándo parar) · c) El modelo será más lento

**5.** `HistGradientBoostingClassifier` **no** tiene equivalente directo de…
- a) `learning_rate` · b) `reg_lambda` · c) `subsample` (muestreo de filas)

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Unos 255 cortes por columna en lugar de uno por valor distinto.
2. **a)** `num_leaves` (y `max_depth` como límite secundario).
3. **b)** *Ordered target statistics*: cada fila se codifica sin ver su propia etiqueta.
4. **b)** Es una fuga sutil: elegiste el número de árboles mirando la prueba (Lección 4.6).
5. **c)** Tiene `l2_regularization` (≈ `reg_lambda`), pero no muestreo de filas.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Si las cuatro librerías dan resultados casi iguales en la mayoría de los datos tabulares, ¿qué criterios usarías para elegir una en un proyecto real?

**B. Caso de análisis 💼** Una cadena de supermercados quiere predecir ventas diarias por **producto** (35,000 productos distintos) y **tienda** (800 tiendas), con 50 millones de filas. Un analista propone hacer one-hot de producto y tienda y usar `GradientBoostingRegressor` de scikit-learn. ¿Qué problemas le ves? ¿Qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Criterios razonables: **qué ya usa tu equipo** (y qué se puede poner en producción), **tamaño de los datos** (LightGBM para muchos millones de filas), **cuántas categorías** hay (CatBoost), si necesitas **GPU**, **cero dependencias** (HistGradientBoosting) y la calidad de la documentación. La diferencia de precisión suele ser menor que la que logras ajustando bien cualquiera de ellas.

**B.** Problemas: one-hot de 35,000 + 800 valores crea ~36,000 columnas casi vacías; `GradientBoostingRegressor` no usa histogramas, así que con 50 millones de filas tardaría días; y los árboles necesitarían miles de preguntas para agrupar productos. Propuesta: **LightGBM** (o CatBoost) con producto y tienda como **categorías nativas**, parada temprana con una validación **temporal** (Lección 4.6: las fechas de validación deben ser posteriores a las de entrenamiento), y quizás variables agregadas (ventas promedio del producto en semanas anteriores).
</details>

### 11.3 🧩 Práctica integradora: ¿quién gana más de 50K?

Con los datos del censo, construirás un boosting moderno de principio a fin y lo compararás con el clásico.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — La línea base: el boosting clásico.** El clásico no acepta categorías ni faltantes, así que hay que prepararle los datos (one-hot + rellenar faltantes). Lo entrenamos con una muestra de 10,000 personas para que no tarde demasiado.

In [ ]:
categoricas = list(Xc_tr.select_dtypes("category").columns)
numericas = [c for c in Xc_tr.columns if c not in categoricas]
preparacion = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categoricas),
    remainder="passthrough")
clasico = make_pipeline(preparacion, GradientBoostingClassifier(random_state=0))

muestra = Xc_tr.sample(10_000, random_state=0).index
inicio = time.perf_counter()
auc_clasico = cross_val_score(clasico, Xc_tr.loc[muestra], yc_tr.loc[muestra], cv=3, scoring="roc_auc").mean()
print(f"Gradient Boosting clásico (10,000 filas): AUC = {auc_clasico:.4f} · {time.perf_counter() - inicio:.1f} s")
print("Columnas después del one-hot:", preparacion.fit_transform(Xc_tr).shape[1])

**Paso 2 🔮 — El moderno, con todos los datos.** HistGradientBoosting con categorías nativas, faltantes nativos y **todos** los datos de entrenamiento. ¿Será más lento o más rápido que el clásico con 10,000 filas? ¿Mejor o peor?

✍️ *Mi apuesta:*

In [ ]:
moderno = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=0)
inicio = time.perf_counter()
auc_moderno = cross_val_score(moderno, Xc_tr, yc_tr, cv=3, scoring="roc_auc").mean()
print(f"HistGradientBoosting ({len(Xc_tr):,} filas): AUC = {auc_moderno:.4f} · {time.perf_counter() - inicio:.1f} s")

<details><summary><b>👀 Ver respuesta</b></summary>

El moderno, con **3.6 veces más datos**, queda mejor en AUC (≈**0.927** contra ≈0.916) y tardó del mismo orden de segundos que el clásico con solo 10,000 filas (en nuestras ejecuciones, ≈2–4 s contra ≈1.5 s). Además no hubo que preparar nada: sin one-hot (el clásico necesitó **104 columnas**) y sin rellenar faltantes. Con el clásico, entrenar con los 36,000 tardaría varias veces más; con millones de filas sería inviable.
</details>

**Paso 3 ✏️ — Ajusta.** Usa lo aprendido en los Ejercicios 6–8: parada temprana con `learning_rate` más bajo y el tamaño de árbol que mejor te funcionó. Completa los valores marcados.

In [ ]:
ajustado = HistGradientBoostingClassifier(
    learning_rate=0.05,        # ✏️
    max_leaf_nodes=31,         # ✏️
    l2_regularization=10.0,    # ✏️
    max_iter=2_000, early_stopping=True, n_iter_no_change=20,
    categorical_features="from_dtype", random_state=0)
auc_ajustado = cross_val_score(ajustado, Xc_tr, yc_tr, cv=3, scoring="roc_auc").mean()
print(f"HistGradientBoosting ajustado: AUC = {auc_ajustado:.4f}  (de fábrica: {auc_moderno:.4f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Con los valores propuestos sale ≈**0.926**, prácticamente igual (incluso una milésima por debajo) que el modelo de fábrica (≈0.927). No es un error tuyo: es lo que ya vimos en el Ejercicio 7. Con datos de este tamaño, el HistGradientBoosting de fábrica (que ya trae parada temprana activada a partir de 10,000 filas) está muy cerca de su techo. ✏️ Prueba otras combinaciones: ganarás o perderás milésimas.

Saber **cuándo dejar de ajustar** también es una habilidad.
</details>

**Paso 4 ▶️ — Examen final, una sola vez.**

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
ajustado.fit(Xc_tr, yc_tr)
prob = ajustado.predict_proba(Xc_te)[:, 1]
print(f"Árboles usados (parada temprana): {ajustado.n_iter_}")
print(f"AUC en prueba      : {roc_auc_score(yc_te, prob):.4f}")
print(f"Exactitud en prueba: {accuracy_score(yc_te, prob >= 0.5):.3f}  (modelo 'nadie gana >50K': {1 - yc_te.mean():.3f})")

✋ ¿Se parece el AUC de prueba al de validación cruzada? ¿Cuánto le gana en exactitud al modelo "tonto"?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

AUC en prueba ≈**0.930**, muy parecido al de validación cruzada (≈0.926): el modelo se comporta en datos nuevos como esperábamos. La parada temprana eligió **377** árboles. En exactitud, ≈**0.875** contra ≈0.761 del modelo "nadie gana más de 50K": unos 11 puntos más, y sobre todo, el modelo "tonto" no identifica a **nadie** del grupo que nos interesa.

⚠️ Recuerda que los datos incluyen sexo, raza y país de origen. Si este modelo se usara para decidir algo sobre personas (un crédito, una oferta de empleo), habría que revisar su comportamiento por grupo (Lección 15.4), como en el proyecto del Módulo 5.
</details>

**Paso 5 ✋ — Recomendación.** Tu jefa pregunta: *"¿Instalamos XGBoost, LightGBM o CatBoost para esto?"*. Responde en tres frases.

<details><summary><b>👀 Ver respuesta</b></summary>

Una respuesta razonable: *"Con 36,000 filas y una docena de columnas, HistGradientBoosting ya entrena en segundos, acepta categorías y faltantes sin preparación y vive dentro de scikit-learn, así que no hace falta nada más. Probaría LightGBM o CatBoost en una tarde para confirmar (suelen quedar a milésimas de AUC), y solo cambiaría si la mejora es clara o si los datos crecen a millones de filas. El tiempo rinde más mejorando los datos y ajustando bien el modelo que cambiando de librería."*
</details>

---
## 📚 Para profundizar
- 📄 **Artículos originales:** Chen y Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*; Ke et al. (2017), *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*; Prokhorenkova et al. (2018), *CatBoost: unbiased boosting with categorical features*.
- 📘 **Documentación:** las páginas de parámetros de XGBoost, LightGBM (*Parameters Tuning*) y CatBoost; y la guía de scikit-learn, sección *Histogram-Based Gradient Boosting*.
- 📺 **Video:** StatQuest, "XGBoost Part 1–4" (inglés, con subtítulos).

---
⬅️ **Anterior:** [6.3 Boosting: aprender de los errores (AdaBoost y Gradient Boosting)](6.3-boosting-adaboost-y-gradient-boosting.ipynb) · ➡️ **Siguiente:** [6.5 Stacking y voting: un comité de modelos distintos](6.5-stacking-y-voting.ipynb)